# Text Generation - LSTM Family Comparison

Notebook này chạy các biến thể LSTM trên đúng một dataset để so sánh kết quả thực nghiệm.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
text = (DATA_DIR / "tiny_shakespeare.txt").read_text(encoding="utf-8")
print(text[:1000])


In [ ]:
text = text[:200000]
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
seq_len = 80
step = 3
sentences, next_chars = [], []
for i in range(0, len(text) - seq_len, step):
    sentences.append(text[i:i+seq_len])
    next_chars.append(text[i+seq_len])

x = np.zeros((len(sentences), seq_len, len(chars)), dtype=np.float32)
y = np.zeros((len(sentences), len(chars)), dtype=np.float32)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_to_idx[char]] = 1.0
    y[i, char_to_idx[next_chars[i]]] = 1.0

x_train, x_test = x[:20000], x[20000:24000]
y_train, y_test = y[:20000], y[20000:24000]

def build_lstm(input_shape):
    return keras.Sequential([layers.Input(shape=input_shape), layers.LSTM(128), layers.Dense(len(chars), activation='softmax')])

def build_stacked_lstm(input_shape):
    return keras.Sequential([layers.Input(shape=input_shape), layers.LSTM(128, return_sequences=True), layers.LSTM(128), layers.Dense(len(chars), activation='softmax')])

def build_bilstm(input_shape):
    return keras.Sequential([layers.Input(shape=input_shape), layers.Bidirectional(layers.LSTM(128)), layers.Dense(len(chars), activation='softmax')])

def build_attention_lstm(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = layers.LSTM(128, return_sequences=True)(inputs)
    x = layers.Attention()([x, x])
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(len(chars), activation='softmax')(x)
    return keras.Model(inputs, outputs)

builders = {
    "StandardLSTM": build_lstm,
    "StackedLSTM": build_stacked_lstm,
    "BiLSTM": build_bilstm,
    "LSTM+Attention": build_attention_lstm,
}

results = []
for name, builder in builders.items():
    model = builder((seq_len, len(chars)))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=128, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results.append({"model": name, "test_accuracy": acc, "test_loss": loss})

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df

sns.barplot(data=results_df, x="test_accuracy", y="model", palette="flare")
plt.title("Text Generation Next-Char Comparison")
plt.show()
